### The Agentic Loop

In agent.ipynb, we did function calling by hand. We sent a message and got back a function call. We ran it, sent the result back, and got the answer.

That works for one function call. It breaks down when the model wants to search several times, or when the first search misses the answer. We don't know in advance how many calls the model will want. So we need a loop that keeps calling the model and running tools until it's done. 

An agent is exactly that.

### Anatomy of an agent
With the LLM in the driver's seat, we have an agent. It's an AI assistant whose goal is to help the user.

An agent has three parts:

- Instructions, the role and behavior we want. We pass this as the developer message. The better the instructions, the better the agent helps.
- Tools, the functions the agent can call to carry out the task. For us that's only search().
- Memory, the message history. We append every prompt, every model output, and every tool result. The agent reads this to know what it has already tried.

### A function-call helper
We'll be running function calls repeatedly inside the loop, so let's wrap that in a small helper. It turns the JSON arguments into a Python dict, calls the right function, and serializes the result. We only have one tool for now, so we dispatch on the function name directly.

In [2]:
import json

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

Duplicate search() for local use in this notebook

In [3]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

The helper returns the exact structure the Responses API expects. When we add more tools later, we'll extend this with more if branches (or switch to a registry).

### Processing one response
Let's process a single model response. We append each output entry to the conversation, print any messages, and run any function calls. Function-call results get appended too.

Instantiate the openai-client...

In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

Build the index...

In [ ]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

Define the search_tool...

In [6]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

### Encouraging multiple searches

The model often answers after the first search, even when more searches would help. It reasons that it already knows enough, so why bother. We push it to explore more by providing the instructions in a developer prompt...

In [1]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

The instructions are how we steer the agent. It can still decide to skip ahead sometimes, so don't expect it to follow them every single run.

In [ ]:
#import json

question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

Response(id='resp_04e08fe4b1ab80e1006a69ead7246c819eb97c3d1e8e8d52dd', created_at=1785326295.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"query":"join course enrollment discovered course can I join"}', call_id='call_K0gnhGxCyh0yHtaktrgEKMhO', name='search', type='function_call', id='fc_04e08fe4b1ab80e1006a69ead85794819eb1061b52f00fb1ab', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"course registration open enrollment late join discovered course"}', call_id='call_tqItM8PUIYiWz2s5kUXe6ZMW', name='search', type='function_call', id='fc_04e08fe4b1ab80e1006a69ead857a8819ea6b178a1a20812cd', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='search', parameters={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query text to 

The has_function_calls flag tells us whether the model needs another API call. If the response contains a function call, the updated messages has tool output the model hasn't seen yet. We'll need to send it back.

### The full agent loop

We wrap this in a while loop. The loop keeps calling the model until it returns a response without any function calls. We also keep an iteration counter so we can see how many round-trips happened.

In [ ]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "deve", "content": instructions},
    {"role": "user", "content": question},
]

max_iterations = 10
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False
    
    print("messages:", messages)

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )
    
    print(response)

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("\nASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False or it > max_iterations:
        break

iteration #1...
messages: [{'role': 'system', 'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."}, {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]
Response(id='resp_0467f956330ef52e006a69edc98df08192bbf79b6e478375d4', created_at=1785327049.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late registration"}', call_id='call_EqqtY7RNypKdo70HxtXLjdRe', name='searc

This is the core agent loop. The model reasons about the next action. Your code performs it, and the model sees the result on the next turn. The loop stops when the model returns a final answer with no more tool calls.

We don't decide how many times the model searches. The model does, and we keep looping until it stops asking for tools.

The exit condition is the simplest one possible. No function calls this turn means we're done. Other frameworks add safety nets on top, like a max iteration count, a token budget, or a wall-clock limit. You might cap it at five iterations and force an answer on the last one. The core is still this one flag.

### Wrapping it in a function

Let's wrap the loop in a function so we can reuse it. The function takes the instructions and the question as parameters, and returns the final answer.

In [14]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

Trying it with a question that has a typo...

In [15]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama local run install local Ollama run locally FAQ"}
function_call: search {"query":"Ollama local setup run model locally FAQ"}
function_call: search {"query":"run Ollama locally terminal command install model FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: download the `.pkg`
     - **Windows**: download the `.msi`
     - **Linux**: run:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a local model**
   ```bash
   ollama run llama3
   ```
   This will download the model, start it locally, and open a chat-like interface.

3. **Check that the server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response showing the models/server info.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```
   Example:
   ```python
   import ollama

   respon

'To run Ollama locally:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - Choose your OS:\n     - **macOS**: download the `.pkg`\n     - **Windows**: download the `.msi`\n     - **Linux**: run:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Start a local model**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model, start it locally, and open a chat-like interface.\n\n3. **Check that the server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response showing the models/server info.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection issue, restarting the server can help:\n```bash\nollama serve\n```\nor

### Restricting off-topic questions

Right now the agent will answer anything. Ask it about chess and it will still try.



In [16]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening Queen's Gambit basic definition"}
function_call: search {"query":"queen gambit course FAQ chess opening"}
function_call: search {"query":"queen's gambit opening 1.d4 d5 2.c4"}
iteration #2...
ASSISTANT:
“Queen gambit” usually means the **Queen’s Gambit**, a chess opening.

It starts with:
1. **d4 d5**
2. **c4**

The idea is that White offers the **c-pawn** to distract Black’s central pawn and gain control of the center. It’s one of the most famous chess openings and is known for strong positional play.

If you want, I can also explain:
- why it’s called a “gambit”
- the main ideas for White and Black
- how to play the Queen’s Gambit as a beginner

Are there other areas you want to explore?


'“Queen gambit” usually means the **Queen’s Gambit**, a chess opening.\n\nIt starts with:\n1. **d4 d5**\n2. **c4**\n\nThe idea is that White offers the **c-pawn** to distract Black’s central pawn and gain control of the center. It’s one of the most famous chess openings and is known for strong positional play.\n\nIf you want, I can also explain:\n- why it’s called a “gambit”\n- the main ideas for White and Black\n- how to play the Queen’s Gambit as a beginner\n\nAre there other areas you want to explore?'

We want a course assistant, not a general chatbot. We tighten the instructions so the agent only answers from the FAQ. For our own use we might be fine letting it answer from general knowledge. So treat this mainly as an illustration of steering scope.

In [17]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [18]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit queen's gambit chess opening"}
iteration #2...
function_call: search {"query":"queen gambit chess opening definition"}
iteration #3...
ASSISTANT:
I couldn’t find any course FAQ entry about the Queen’s Gambit, so it looks like this is off-topic for the course materials.

If you want, I can still help with a course-related question. Is there another area you want to explore?


'I couldn’t find any course FAQ entry about the Queen’s Gambit, so it looks like this is off-topic for the course materials.\n\nIf you want, I can still help with a course-related question. Is there another area you want to explore?'

This is a lightweight form of an input guardrail. We tell the agent what's in scope and what isn't. A real guardrail checks the input before the agent runs and can block off-topic questions outright. That's a separate topic, but instructions are the first place to start.

This handwritten loop is the best way to understand what frameworks hide from you. Every agent framework wraps this same pattern, whether it's LangChain, PydanticAI, or the OpenAI Agents SDK.